# Import Required Libraries
This section imports arcpy, arcgis.gis, and arcgis.features libraries for GIS operations.

In [17]:
import arcpy, os
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

# Connect to GIS and Query Features
Connect to the active ArcGIS Pro session and query features from the specified feature layer.

In [18]:
gis = GIS("pro")
layer_url = "https://services6.arcgis.com/sIGsGHq6XlKtUAL8/arcgis/rest/services/CA_LB_PRO/FeatureServer/3"
layer = FeatureLayer(layer_url, gis=gis)
where_clause = "1=1"
geometry = {
    "xmin": -13417214.5529552419,
    "ymin": 3919744.3389966148,
    "xmax": -12705061.7473576926,
    "ymax": 4636886.3236006815,
    "spatialReference": {"wkid": 102100}
}
result = layer.query(
    where=where_clause,
    geometry=geometry,
    geometry_type='esriGeometryEnvelope',
    spatial_rel='esriSpatialRelIntersects',
    out_fields="OBJECTID",
    return_geometry=True,
    result_offset=16000,
    result_record_count=4,
    out_sr=102100,
    return_true_curves=True,
    return_pbf=True
)
print(f"Returned {len(result.features)} features.")

Returned 4 features.


# Convert Query Result to Feature Class
Convert the query result to a feature class in the current workspace using arcpy.

In [19]:
arcpy.env.workspace = arcpy.env.workspace or arcpy.management.GetWorkspace()
print(arcpy.env.workspace)
output_fc = "query_result"
if arcpy.Exists(output_fc):
    arcpy.management.Delete(output_fc)
# Save the FeatureSet to a temporary JSON file
result_json = "result.json"
result.save(r"c:\temp",result_json)
arcpy.conversion.JSONToFeatures(os.path.join(r"c:\temp",result_json), output_fc)
print(f"Feature class '{output_fc}' created in workspace: {arcpy.env.workspace}")

C:\Users\mcveydb\Projects\parcels\Default.gdb
Feature class 'query_result' created in workspace: C:\Users\mcveydb\Projects\parcels\Default.gdb


# Add Feature Class to Open Map
Add the new feature class to the active map in ArcGIS Pro.

In [ ]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
active_map = aprx.activeMap
active_map.addDataFromPath(arcpy.env.workspace + "\\" + output_fc)
print(f"Added '{output_fc}' to the active map.")